In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/career-con-2019/sample_submission.csv
/kaggle/input/competitions/career-con-2019/X_test.csv
/kaggle/input/competitions/career-con-2019/y_train.csv
/kaggle/input/competitions/career-con-2019/X_train.csv


In [2]:
path = '/kaggle/input/competitions/career-con-2019/'
X_train = pd.read_csv(path+'X_train.csv')
test = pd.read_csv(path+'X_test.csv')
sub = pd.read_csv(path+'sample_submission.csv')
y_train = pd.read_csv(path+'y_train.csv')

In [3]:
print(X_train.groupby('series_id').size().value_counts())
print(y_train['surface'].unique())
#print(X_train.isnull().sum())

128    3810
Name: count, dtype: int64
['fine_concrete' 'concrete' 'soft_tiles' 'tiled' 'soft_pvc'
 'hard_tiles_large_space' 'carpet' 'hard_tiles' 'wood']


128시리즈가 한 시계열


surface는 9개 클래스 문자열


In [4]:
X_train.head()

,row_id,series_id,measurement_number,orientation_X,orientation_Y,orientation_Z,orientation_W,angular_velocity_X,angular_velocity_Y,angular_velocity_Z,linear_acceleration_X,linear_acceleration_Y,linear_acceleration_Z
0,0_0,0,0,-0.75853,-0.63435,-0.10488,-0.10597,0.107650,0.017561,0.000767,-0.74857,2.1030,-9.7532
1,0_1,0,1,-0.75853,-0.63434,-0.10490,-0.10600,0.067851,0.029939,0.003386,0.33995,1.5064,-9.4128
2,0_2,0,2,-0.75853,-0.63435,-0.10492,-0.10597,0.007275,0.028934,-0.005978,-0.26429,1.5922,-8.7267
3,0_3,0,3,-0.75852,-0.63436,-0.10495,-0.10597,-0.013053,0.019448,-0.008974,0.42684,1.0993,-10.0960
4,0_4,0,4,-0.75852,-0.63435,-0.10495,-0.10596,0.005135,0.007652,0.005245,-0.50969,1.4689,-10.4410


In [5]:
test.head()

,row_id,series_id,measurement_number,orientation_X,orientation_Y,orientation_Z,orientation_W,angular_velocity_X,angular_velocity_Y,angular_velocity_Z,linear_acceleration_X,linear_acceleration_Y,linear_acceleration_Z
0,0_0,0,0,0.91208,-0.38193,-0.050618,0.14028,-0.060205,0.071286,-0.18787,0.29492,2.8027,-9.6816
1,0_1,0,1,0.91220,-0.38165,-0.050573,0.14028,-0.033486,0.060210,-0.18206,0.14944,2.5408,-9.8521
2,0_2,0,2,0.91228,-0.38143,-0.050586,0.14032,-0.029686,0.029476,-0.18441,-0.49741,2.5853,-9.3835
3,0_3,0,3,0.91237,-0.38121,-0.050588,0.14035,-0.024217,0.037788,-0.18783,-0.32376,2.9966,-8.7415
4,0_4,0,4,0.91247,-0.38096,-0.050546,0.14042,-0.038047,0.083405,-0.20170,-0.70103,2.6498,-8.8432


In [6]:
sub.head()

,series_id,surface
0,0,concrete
1,1,concrete
2,2,concrete
3,3,concrete
4,4,concrete


In [7]:
y_train.head()

,series_id,group_id,surface
0,0,13,fine_concrete
1,1,31,concrete
2,2,20,concrete
3,3,31,concrete
4,4,22,soft_tiles


In [8]:
print(y_train.groupby('group_id').size())

group_id
0     57
1     38
2     18
3     57
4     57
      ..
68    70
69    70
70    71
71    70
72    70
Length: 73, dtype: int64


group_id : ID number for all of the measurements taken in a recording session. Provided for the training set only, to enable more cross validation strategies.

In [9]:
print(y_train.groupby('group_id')['surface'].nunique().max())
print(y_train.groupby('surface')['group_id'].nunique())

1
surface
carpet                     4
concrete                  15
fine_concrete              7
hard_tiles                 1
hard_tiles_large_space     5
soft_pvc                  14
soft_tiles                 6
tiled                      9
wood                      12
Name: group_id, dtype: int64


In [10]:
print(y_train['surface'].value_counts(normalize=True))

surface
concrete                  0.204462
soft_pvc                  0.192126
wood                      0.159318
tiled                     0.134908
fine_concrete             0.095276
hard_tiles_large_space    0.080840
soft_tiles                0.077953
carpet                    0.049606
hard_tiles                0.005512
Name: proportion, dtype: float64


In [11]:
def make_features(df):
    df = df.sort_values(['series_id','measurement_number'])
    sensors = df.columns[3:]
    features = df.groupby('series_id')[sensors].agg(['min','max','std','skew','median', 'mean'])
    features.columns = [f'{s}_features_{stat}' for s, stat in features.columns]

    diff_features = df[sensors].diff().where(df['measurement_number']!=0)
    diff_features = diff_features.groupby(df['series_id'])[sensors].agg(['min','max','std'])
    diff_features.columns = [f'{s}_diff_features_{stat}' for s, stat in diff_features.columns]
    #나머지는 하는 중..

    features = features.join(diff_features, rsuffix='_features')
    return features

In [12]:
from sklearn.preprocessing import LabelEncoder

X = make_features(X_train)

y = y_train.set_index('series_id').loc[X.index, 'surface']
LE = LabelEncoder()
y = pd.Series(LE.fit_transform(y), index=X.index) #결과는 numpy배열이라 series..

groups = y_train.set_index('series_id').loc[X.index, 'group_id']

X_test = make_features(test).loc[sub['series_id']]
X_test.columns = X.columns

In [13]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score

gkf = GroupKFold(n_splits=5)
oof = np.zeros([len(X), len(LE.classes_)])
test_pred = np.zeros([len(X_test), len(LE.classes_)])

for fold, (tr_idx, ev_idx) in enumerate(gkf.split(X,y,groups)):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, verbose=-1)

    tr_classes = np.unique(y.iloc[tr_idx])                    # 이 폴드 학습에 있는 클래스
    mask = y.iloc[ev_idx].isin(tr_classes).values             # 검증 중 아는 클래스만 True

    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
          eval_set=[(X.iloc[ev_idx][mask], y.iloc[ev_idx][mask])],
          callbacks=[lgb.early_stopping(100, verbose=False)])

    cls = model.classes_                                        
    oof[np.ix_(ev_idx, cls)] = model.predict_proba(X.iloc[ev_idx])
    test_pred[:, cls] += model.predict_proba(X_test) / gkf.n_splits
    #oof[ev_idx] = model.predict_proba(X.iloc[ev_idx]) #9개 클래스
    #test_pred += model.predict_proba(X_test)/gkf.n_splits
    print(f'Fold {fold} Accuracy: {accuracy_score(y.iloc[ev_idx],oof[ev_idx].argmax(axis=1)):.4f}')

print(f'Accuracy: {accuracy_score(y, oof.argmax(axis=1)):.4f}')

Fold 0 Accuracy: 0.4507
Fold 1 Accuracy: 0.4226
Fold 2 Accuracy: 0.4653
Fold 3 Accuracy: 0.4042
Fold 4 Accuracy: 0.3005
Accuracy: 0.4087


In [14]:
sub['surface'] = LE.inverse_transform(test_pred.argmax(axis=1))
sub.to_csv('submission.csv', index=False)
print(sub.head())

   series_id   surface
0          0  soft_pvc
1          1  concrete
2          2  concrete
3          3      wood
4          4  concrete
